<!-- Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. -->

# MJX 09 — Playground PPO: Franka Panda pick-and-place

The capstone manipulation task: **`PandaPickCube`** — a 7-DoF Franka Panda arm
that must reach, grasp, and lift a cube. This is a real robot arm from the
MuJoCo Menagerie, so it is far harder (and easier to destabilise) than the DM
Control toy tasks.

## Why this notebook is tuned differently
With the stock high learning rate, PPO on this task **diverges to NaN** on the
gfx1151 APU. We trade speed for stability — learning is slower but never blows
up — using:
- **`learning_rate = 5e-5`** (down from the default) and **`max_grad_norm = 1.0`** (gradient clipping)
- **`normalize_observations = True`** and **`reward_scaling = 0.1`**
- **`num_envs = 256`** + highest matmul precision (same GPU-safety knobs as MJX 07/08)

Expect the eval reward to climb gradually — "學得慢沒關係", the point is a stable curve with no NaNs.

In [ ]:
import os, functools
os.environ["MUJOCO_GL"] = "egl"

import jax
import jax.numpy as jp
if not hasattr(jax, "device_put_replicated"):
    def _dpr(x, devices=None):
        n = len(devices) if devices is not None else jax.local_device_count()
        return jax.tree_util.tree_map(
            lambda a: jax.device_put(jp.broadcast_to(jp.asarray(a)[None], (n,) + jp.asarray(a).shape)), x)
    jax.device_put_replicated = _dpr
jax.config.update("jax_default_matmul_precision", "highest")

import numpy as np
import matplotlib.pyplot as plt
import imageio
from mujoco_playground import registry, wrapper
from mujoco_playground._src import mjx_env
from mujoco_playground.config import manipulation_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from IPython.display import Video

# The Panda model + assets live in the MuJoCo Menagerie. The image pre-downloads
# them at build time; this is a no-op safety net if they are already present.
try:
    mjx_env.ensure_menagerie_exists()
except Exception as e:
    print("menagerie check:", e)

print("JAX devices:", jax.devices())

In [ ]:
ENV_NAME = "PandaPickCube"
env = registry.load(ENV_NAME, config_overrides={"impl": "jax"})

cfg = manipulation_params.brax_ppo_config(ENV_NAME)
ppo_params = cfg.to_dict()
net_cfg = ppo_params.pop("network_factory", None)
# Stability-first overrides (see notebook header).
ppo_params.update(
    num_timesteps=4_000_000,
    num_envs=256,
    batch_size=256,
    num_minibatches=8,
    num_evals=10,
    learning_rate=5e-5,
    max_grad_norm=1.0,
    normalize_observations=True,
    reward_scaling=0.1,
    clipping_epsilon=0.2,
)
print({k: ppo_params[k] for k in ["num_timesteps", "num_envs", "learning_rate", "max_grad_norm"]})

In [ ]:
progress = []
def progress_fn(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0))
    progress.append((int(step), r))
    flag = "  <-- NaN!" if np.isnan(r) else ""
    print(f"step {int(step):>9}  eval reward {r:8.2f}{flag}")

train_fn = functools.partial(ppo.train, **ppo_params)
if net_cfg:
    train_fn = functools.partial(train_fn,
        network_factory=functools.partial(ppo_networks.make_ppo_networks, **net_cfg))

make_inference_fn, params, _ = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
    progress_fn=progress_fn,
    seed=0,
)
print("training done")

In [ ]:
steps, rewards = zip(*progress)
plt.figure(figsize=(7, 3))
plt.plot(steps, rewards, marker="o")
plt.xlabel("environment steps"); plt.ylabel("eval episode reward")
plt.title(f"PPO learning curve ({ENV_NAME}) — stabilised"); plt.grid(True); plt.show()

In [ ]:
os.makedirs("output/videos", exist_ok=True)
inference = jax.jit(make_inference_fn(params))
reset, step = jax.jit(env.reset), jax.jit(env.step)

rng = jax.random.PRNGKey(1)
state = reset(rng)
trajectory = [state]
for _ in range(150):
    rng, k = jax.random.split(rng)
    action, _ = inference(state.obs, k)
    state = step(state, action)
    trajectory.append(state)

frames = np.asarray(env.render(trajectory, height=240, width=320))
out = "output/videos/mjx09_panda_pick_cube.mp4"
imageio.mimsave(out, list(frames), fps=30)
print("saved", frames.shape[0], "frames ->", out)

In [ ]:
Video(url="output/videos/mjx09_panda_pick_cube.mp4")